# docextract: heavy evaluation on Colab
Runs the OCR engines on the public gold sets (FUNSD forms, scanned arXiv pages). Results are written one file at a time to Google Drive, so a disconnect loses at most one file; rerun the cell to resume.

**Before running:** Runtime > Change runtime type > GPU (T4 or better). Upload the project folder (without `.venv`, `data/public`, `outputs/cache`) to Drive at `MyDrive/document-capture`, or push it to a Git repo and clone it below.

In [ ]:
from google.colab import drive
drive.mount('/content/drive')
%cd /content/drive/MyDrive/document-capture

In [ ]:
!apt-get -qq install -y tesseract-ocr > /dev/null
!pip -q install pdfplumber pydantic pytesseract rapidocr-onnxruntime onnxruntime-gpu opencv-python-headless pandas rapidfuzz

## 1. Build the gold sets (downloads FUNSD + arXiv PDFs)

In [ ]:
import os, subprocess
os.makedirs('data/public/arxiv', exist_ok=True)
if not os.path.exists('data/public/funsd_raw/dataset'):
    !curl -sL -o data/public/funsd.zip https://guillaumejaume.github.io/FUNSD/dataset.zip
    !unzip -q -o data/public/funsd.zip -d data/public/funsd_raw
for pid in "1706.03762 1810.04805 1512.03385 1409.1556 1412.6980 1502.03167 2010.11929 1706.02677".split():
    if not os.path.exists(f'data/public/arxiv/{pid}.pdf'):
        !curl -sL -o data/public/arxiv/{pid}.pdf https://arxiv.org/pdf/{pid}
        !sleep 3
!python scripts/make_gold.py

## 2. Run the engines (resumable)

In [ ]:
for g in ['funsd', 'arxiv']:
    for e in ['rapidocr', 'tesseract']:
        !python scripts/eval_extract.py data/gold/{g} --engine {e}

## 3. Results

In [ ]:
import glob, pandas as pd
rows = []
for f in sorted(glob.glob('outputs/results/extract_*.csv')):
    d = pd.read_csv(f); name = f.split('extract_')[1][:-4]
    rows.append(dict(run=name, files=len(d), f1=d.f1.mean(), recall=d.recall.mean(), cer=d.cer.mean(), sec_per_file=d.seconds.mean()))
pd.DataFrame(rows).round(3)

## 4. Next: layout and table models (GPU)
Add Docling and PP-StructureV3 adapters in `docextract/engines.py`, then rerun section 2 with `--engine docling` etc. Table quality should be scored with TEDS on a PubTabNet subset, which is not built yet.